# Ready Dataset Review

Run this notebook with the bundled CSVs; videos are not required. These checks validate annotations, not media playback. Original-video separation is not participant separation.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# Detect the repository from its root or notebooks directory.
project = Path.cwd()
if not (project / "dataset/csvs").is_dir():
    project = project.parent
if not (project / "dataset/csvs").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder.")
csv_folder = project / "dataset/csvs"


In [ ]:
from itertools import combinations

# Load supplied annotations and validate labels and unique clip IDs.
frames = {
    name: pd.read_csv(csv_folder / f"{name}_ready.csv")
    for name in ("dataset", "train", "val", "test")
}
labels = [c for c in frames["dataset"].columns if c != "Video_ID"]
expected_counts = {"dataset": 527, "train": 312, "val": 108, "test": 107}
for name, frame in frames.items():
    assert list(frame.columns) == list(frames["dataset"].columns)
    assert len(frame) == expected_counts[name], f"Unexpected count: {name}"
    assert not frame.isna().any().any(), f"Missing values: {name}"
    assert not frame["Video_ID"].duplicated().any(), f"Duplicate IDs: {name}"
    assert frame[labels].isin([0, 1]).all().all(), f"Invalid labels: {name}"
    print(f"{name}: {len(frame)} rows; labels and IDs valid")

# Verify both clip and original-source separation.
for a, b in combinations(("train", "val", "test"), 2):
    ids_a = set(frames[a]["Video_ID"])
    ids_b = set(frames[b]["Video_ID"])
    assert not ids_a & ids_b
    assert not {v.rsplit("_", 2)[0] for v in ids_a} & {v.rsplit("_", 2)[0] for v in ids_b}
    print(f"{a}/{b}: clip overlap=0, source overlap=0")

# Compare complete row contents, including every label.
combined = pd.concat([frames[s] for s in ("train", "val", "test")], ignore_index=True)
pd.testing.assert_frame_equal(
    frames["dataset"].set_index("Video_ID").sort_index(),
    combined.set_index("Video_ID").sort_index(),
)
report = pd.read_csv(csv_folder / "video_check_report.csv")
assert not report["Video_ID"].duplicated().any()
valid_ids = set(report.loc[report["status"] == "metadata_ok", "Video_ID"])
assert valid_ids == set(frames["dataset"]["Video_ID"])
print("Report coverage and all split labels match.")
print(report["status"].value_counts().to_string())


In [ ]:
# Compare positive label counts across the ready splits.
label_counts = pd.DataFrame({
    name: frames[name][labels].sum()
    for name in ("train", "val", "test")
})
print(label_counts.to_string())
print("\nPositive label percentage across the ready dataset:")
print((frames["dataset"][labels].mean() * 100).round(2).to_string())


In [ ]:
import matplotlib.pyplot as plt

# Display the label distribution; multiple labels may belong to one clip.
frames["dataset"][labels].sum().sort_values().plot.barh(figsize=(10, 6))
plt.xlabel("Clips with a positive label")
plt.title("Available AV-ASD Subset: Label Distribution")
plt.tight_layout()
plt.show()
